# Wikipedia Web Scraping: Renewable Electricity & Government Debt

**Group 4**

- Yaroslav Kuzmin
- Yevhen Nezghovorov
- Hlieb Nikoniuk
- Tymur Tleuberlin

<br>



- **Table 1:** [List of countries by renewable electricity production](https://en.wikipedia.org/wiki/List_of_countries_by_renewable_electricity_production) (Location, Renew., Hydro, Wind, Solar, Bio., Other)
- **Table 2:** [List of countries by government debt](https://en.wikipedia.org/wiki/List_of_countries_by_government_debt) (Country, Gross debt % of GDP 2022)

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from io import StringIO

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

url1 = "https://en.wikipedia.org/wiki/List_of_countries_by_renewable_electricity_production"
url2 = "https://en.wikipedia.org/wiki/List_of_countries_by_government_debt"

headers = {
    'User-Agent': 'DataAnalysisBot/1.0 (Educational purposes; contact@example.com)'
}

response = requests.get(url1, headers=headers)
response.raise_for_status()
tables = pd.read_html(StringIO(response.text))
df_renew = tables[1].copy()

response = requests.get(url2, headers=headers)
response.raise_for_status()
tables = pd.read_html(StringIO(response.text))
df_debt = tables[0].copy()

#----- cleaning tables

# handle multi-level headers
df_renew.columns = [str(c) for c in df_renew.columns]
df_debt.columns = [str(c) for c in df_debt.columns]

# remove strange signs from column names (e.g. [1], [citation needed])
df_renew.columns = df_renew.columns.str.replace(r'\[.*\]', '', regex=True).str.strip()
df_debt.columns = df_debt.columns.str.replace(r'\[.*\]', '', regex=True).str.strip()


# rename the first column to 'Country' as it is foreign key for this two tables and should be the same
df_renew.rename(columns={df_renew.columns[0]: 'Country'}, inplace=True)
df_debt.rename(columns={df_debt.columns[0]: 'Country'}, inplace=True)

# remove summary row from renew table so they will be identical
df_renew = df_renew[df_renew['Country'] != 'World']


# keep only Country and the 2022 Debt column (column name may vary by Wikipedia layout)
candidates = [c for c in df_debt.columns if '2022' in str(c) or 'Gross debt' in str(c)]
target_debt_col = candidates[0] if candidates else df_debt.columns[2]
df_debt = df_debt[['Country', target_debt_col]]
df_debt.rename(columns={target_debt_col: 'Gov_Debt_Percent_GDP_2022'}, inplace=True)

# removes footnotes like 'China[1]' -> 'China'
def clean_country_name(name):
    return re.sub(r'\[.*?\]', '', str(name)).strip()

# removes commas, %, and non-numeric chars, then converts to float
def clean_number(x):
    if pd.isna(x): return x
    x = str(x).replace(',', '').replace('%', '')
    match = re.search(r'-?\d+(\.\d+)?', x)
    return float(match.group()) if match else None

# apply cleaning
for df in [df_renew, df_debt]:
    df['Country'] = df['Country'].apply(clean_country_name)

# normalize country names so merge matches (debt table uses different spellings)
debt_to_standard = {
    'Democratic Republic of the Congo': 'DR Congo',
    'Republic of the Congo': 'Congo',
    "Côte d'Ivoire": 'Ivory Coast',
    'Georgia (country)': 'Georgia',
    'Kyrgyz Republic': 'Kyrgyzstan',
}
df_debt['Country'] = df_debt['Country'].replace(debt_to_standard)

# convert numerical columns in Renewables
cols_to_fix = df_renew.columns[1:7]
for col in cols_to_fix:
    df_renew[col] = df_renew[col].apply(clean_number)

# convert numerical column in Debt
df_debt['Gov_Debt_Percent_GDP_2022'] = df_debt['Gov_Debt_Percent_GDP_2022'].apply(clean_number)



#----- merge table


# keep countries that appear in both tables
merged_df = pd.merge(df_renew, df_debt, on='Country', how='inner')

# drop rows with missing values
merged_df.dropna(subset=['Renew.', 'Gov_Debt_Percent_GDP_2022'], inplace=True)

# data overview (30+ countries required)
print("Merged dataset shape:", merged_df.shape)
print("Countries in merged data:", len(merged_df))
merged_df.head(10)


# ----- Visualizations (4 total: 2 maps + 2 other charts) -----

# 1. Map: Government Debt (% of GDP, 2022)
fig1 = px.choropleth(merged_df,
                     locations="Country",
                     locationmode='country names',
                     color="Gov_Debt_Percent_GDP_2022",
                     hover_name="Country",
                     color_continuous_scale=px.colors.sequential.Reds,
                     title="Global Government Debt (% of GDP, 2022)")
fig1.update_geos(showcountries=True, coastlinecolor='lightgray',
                 showland=True, landcolor='white',
                 projection_type='natural earth', showframe=True)
fig1.update_layout(height=420, margin=dict(l=0, r=0, t=40, b=0), font=dict(size=11))
fig1.show()

# 2. Map: Renewable Electricity Production (TWh)
fig2 = px.choropleth(merged_df,
                     locations="Country",
                     locationmode='country names',
                     color="Renew.",
                     hover_name="Country",
                     color_continuous_scale=px.colors.sequential.Viridis,
                     range_color=(0, 500),
                     title="Renewable Electricity Production (TWh, 2023)")
fig2.update_geos(showcountries=True, coastlinecolor='lightgray',
                 showland=True, landcolor='white',
                 projection_type='natural earth', showframe=True)
fig2.update_layout(height=420, margin=dict(l=0, r=0, t=40, b=0), font=dict(size=11))
fig2.show()

# 3. Scatter: Government Debt vs. Renewable Production (size = Hydro)
fig3 = px.scatter(merged_df,
                  x="Gov_Debt_Percent_GDP_2022",
                  y="Renew.",
                  size="Hydro",
                  hover_name="Country",
                  title="Government Debt vs. Renewable Production (Size = Hydro Output)",
                  labels={"Gov_Debt_Percent_GDP_2022": "Gov Debt % (2022)", "Renew.": "Renewable Output (TWh)"})
fig3.show()

# 4. Bar chart: Top 10 Wind Energy Producers and Debt Levels
top_wind = merged_df.sort_values(by='Wind', ascending=False).head(10)

fig4 = px.bar(top_wind,
              x='Country',
              y='Wind',
              color='Gov_Debt_Percent_GDP_2022',
              title="Top 10 Wind Energy Producers and their Debt Levels",
              labels={'Wind': 'Wind Production (TWh)', 'Gov_Debt_Percent_GDP_2022': 'Debt % of GDP'})
fig4.show()


Merged dataset shape: (180, 9)
Countries in merged data: 180


## Brief findings

- **Government debt (map 1):** Japan and several European countries(Greece, Italy, France etc.), have debt above 100% of GDP. Many emerging economies and oil exporters have lower ratios. The EU 60% benchmark is exceeded in many countries.

- **Renewable electricity (map 2):** China, the US, Brazil, Canada, and India lead in total renewable generation (TWh). This mainly reflects country size and strong hydro, wind, or solar capacity. Norway and Brazil stand out for a high renewable share relative to size.

- **Debt vs renewables (scatter):** There is no clear linear relationship between public debt and renewable energy output. Large renewable producers are at both low and high debt levels. Hydro-based systems often show high total renewable generation.

- **Top wind producers (bar chart):** China and the US lead in wind generation. Germany, the UK, Spain, and France are also among the top. Debt levels vary widely across these countries, suggesting no direct link between wind output and government debt.